# Notebook 07: Web Application Asset Export

**Milestone:** M8  
**Purpose:** Generate every static asset consumed by `app/` (the Conserved Aging Explorer) directly from `results/*.csv` golden outputs. No number in `app/data` is typed by hand: each value is read from a `results/*.csv` file in this notebook, and `app/data/manifest.json` records, for every asset, its source CSV path(s) plus the SHA-256 of each source file.

**Budgets (binding):** every individual asset file ≤ 1 MB; total under `app/data/` ≤ 5 MB. The pre-registered hosting budgets (≤ 25 MB per view, ≤ 300 MB total) remain the outer ceilings, but the real data at this scale fits in < 1 MB — see `notes/app-stack.md`.

**Determinism:** `src.set_seeds()` (seed 42) at top, per repo convention. The UMAP subsample uses `numpy.random.default_rng(42)` with a per-tissue proportional allocation capped at 5,000 cells total; the seed and the per-tissue counts are recorded inside the asset and in `manifest.json`.

**Inputs:** `results/*.csv` (this notebook reads them; nothing is hardcoded), `data/interim/m1-processed.h5ad` (obs + `obsm['X_umap']` only, backed mode).

**Outputs:** `app/data/*.json`, `app/data/manifest.json`, `app/figures/*.png` (copies of `figures/*.png`).

In [1]:
import os
import sys
import json
import shutil
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))
from src.set_seeds import set_seeds

set_seeds(42)  # repo convention: deterministic execution

RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
APP_DIR = REPO_ROOT / "app"
APP_DATA_DIR = APP_DIR / "data"
APP_FIGURES_DIR = APP_DIR / "figures"
H5AD_PATH = REPO_ROOT / "data" / "interim" / "m1-processed.h5ad"

APP_DATA_DIR.mkdir(parents=True, exist_ok=True)
APP_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

MAX_ASSET_BYTES = 1_048_576        # 1 MB per individual asset file
MAX_DATA_TOTAL_BYTES = 5_242_880   # 5 MB total under app/data
UMAP_SEED = 42
UMAP_CAP = 5000

def sha256_of(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def read_results_csv(name: str) -> pd.DataFrame:
    """Read a golden results CSV; fail loudly if missing (never fabricate)."""
    p = RESULTS_DIR / name
    assert p.exists(), f"required results file missing: {p}"
    return pd.read_csv(p)

def clean_records(df: pd.DataFrame) -> list:
    """DataFrame -> JSON-safe records; NaN -> None. Full float precision is preserved
    (df.to_json would truncate to 10 significant digits, corrupting the golden numbers)."""
    records = []
    for row in df.itertuples(index=False):
        rec = {}
        for col, v in zip(df.columns, row):
            if pd.isna(v):
                rec[col] = None
            elif isinstance(v, np.integer):
                rec[col] = int(v)
            elif isinstance(v, np.floating):
                rec[col] = float(v)
            elif isinstance(v, (np.bool_, bool)):
                rec[col] = bool(v)
            else:
                rec[col] = v
        records.append(rec)
    return records

# Registry of everything this notebook writes. Keyed by asset name; filled by write_* calls.
ASSETS = {}

def register_asset(key, rel_path: str, description: str, source_paths: list):
    ASSETS[key] = {
        "path": rel_path,
        "description": description,
        "sources": [{"path": str(p.relative_to(REPO_ROOT)), "sha256": sha256_of(p)} for p in source_paths],
    }

def write_json_asset(key: str, payload: dict, description: str, source_names: list):
    source_paths = [RESULTS_DIR / n for n in source_names]
    for p in source_paths:
        assert p.exists(), f"source CSV missing for asset '{key}': {p}"
    out = APP_DATA_DIR / f"{key}.json"
    with open(out, "w") as f:
        json.dump(payload, f)
    size = out.stat().st_size
    assert size <= MAX_ASSET_BYTES, f"asset '{key}' is {size} bytes > {MAX_ASSET_BYTES}"
    register_asset(key, f"data/{key}.json", description, source_paths)
    print(f"  wrote {out.relative_to(REPO_ROOT)} ({size/1024:.1f} KB) <- {source_names}")

# Remove stale assets from the previous (pre-real-data) export so nothing fabricated survives.
stale = [p for p in APP_DATA_DIR.glob("*.json")]
for p in stale:
    p.unlink()
    print(f"  removed stale {p.relative_to(REPO_ROOT)}")
print(f"ASSET BUDGETS: {MAX_ASSET_BYTES/1024:.0f} KB/file, {MAX_DATA_TOTAL_BYTES/1024/1024:.0f} MB total under app/data")

  removed stale app/data/ranking.json
  removed stale app/data/program_genes.json
  removed stale app/data/savina.json
  removed stale app/data/agreement.json
  removed stale app/data/gene_programs.json
  removed stale app/data/signature.json
  removed stale app/data/manifest.json
  removed stale app/data/confound.json
  removed stale app/data/umap_subsample.json
ASSET BUDGETS: 1024 KB/file, 5 MB total under app/data


## 1. Signature view asset — longevity-outlier contrast (M6)

Sources: `results/m6-statistical-summary.csv` (slopes + CIs + honest p-values, incl. the failed TOST), `results/m6-lifespan-program-scores.csv` (sample-level program scores), `results/m6-species-lifespans.csv` (AnAge denominators), `results/m6-gates.csv` (positive-control gate + program-gene coverage per species).

In [2]:
stats_df = read_results_csv("m6-statistical-summary.csv")
scores_df = read_results_csv("m6-lifespan-program-scores.csv")
lifespans_df = read_results_csv("m6-species-lifespans.csv")
gates_df = read_results_csv("m6-gates.csv")

signature = {
    "statistics": clean_records(stats_df),
    "scores": clean_records(scores_df),
    "lifespans": clean_records(lifespans_df),
    "gates": clean_records(gates_df),
}
write_json_asset(
    "signature", signature,
    "M6 longevity-outlier contrast: per-sample conserved-program scores, per-species OLS slopes with 95% CIs and p-values (incl. TOST equivalence and exact permutation), AnAge lifespan denominators, and M6 gate values (positive control, program-gene coverage).",
    ["m6-statistical-summary.csv", "m6-lifespan-program-scores.csv", "m6-species-lifespans.csv", "m6-gates.csv"],
)

  wrote app/data/signature.json (9.4 KB) <- ['m6-statistical-summary.csv', 'm6-lifespan-program-scores.csv', 'm6-species-lifespans.csv', 'm6-gates.csv']


## 2. Savina 2022 reproduction asset (M6)

Sources: `results/m6-savina-reproduction.csv` (NMR epidermis DE-count vs |log2FC| cutoff curve), `results/m2-calibration-positive-controls.csv` (canonical aging-control genes in the NMR contrast: Igfbp3, Il6, Apoe, Lgals3).

In [3]:
savina_curve_df = read_results_csv("m6-savina-reproduction.csv")
pos_ctrl_df = read_results_csv("m2-calibration-positive-controls.csv")

savina = {
    "curve": clean_records(savina_curve_df),
    "positive_controls": clean_records(pos_ctrl_df),
    "note": ("Igfbp3 — the gene behind Savina 2022's within-cluster aged-vs-young claim — is absent "
             "from the processed GSE152886 matrix (no results/m6-igfbp3.csv was producible; see "
             "scripts/run_m6_pipeline.py), so its NMR behavior is untestable here. The rows above are "
             "the control genes as computed in results/m2-calibration-positive-controls.csv."),
}
write_json_asset(
    "savina", savina,
    "Savina 2022 NMR reproduction: DE-count vs |log2FC| cutoff curve (aged vs young, NMR epidermis) and canonical aging-control gene results.",
    ["m6-savina-reproduction.csv", "m2-calibration-positive-controls.csv"],
)

  wrote app/data/savina.json (3.2 KB) <- ['m6-savina-reproduction.csv', 'm2-calibration-positive-controls.csv']


## 3. Conserved aging program gene sets (M6)

Source: `results/m6-program-genes.csv` — the conserved program gene lists (up / down / conserved_up / conserved_down).

In [4]:
prog_df = read_results_csv("m6-program-genes.csv")
program_genes = {
    "program_up": prog_df["program_up"].dropna().tolist(),
    "program_down": prog_df["program_down"].dropna().tolist(),
    "conserved_up": prog_df["conserved_up"].dropna().tolist(),
    "conserved_down": prog_df["conserved_down"].dropna().tolist(),
    "counts": {
        "n_program_up": int(prog_df["program_up"].notna().sum()),
        "n_program_down": int(prog_df["program_down"].notna().sum()),
        "n_conserved_up": int(prog_df["conserved_up"].notna().sum()),
        "n_conserved_down": int(prog_df["conserved_down"].notna().sum()),
    },
}
write_json_asset(
    "program_genes", program_genes,
    "M6 conserved aging program gene lists: up-regulated (with age), down-regulated, and the strictly conserved subsets.",
    ["m6-program-genes.csv"],
)

  wrote app/data/program_genes.json (2.6 KB) <- ['m6-program-genes.csv']


## 4. Mouse aging-signal ranking asset (M2)

Sources: `results/m2-aging-signal-ranked.csv` (the 17 powered units, ranked, with bootstrap CIs), `results/m2-unit-eligibility.csv` (163 units evaluated → 17 powered; eligibility rule ≥3 mice and ≥50 cells per group), `results/m2-calibration-trio-summary.csv` (label-permutation null + sex-effect floor for the top unit), `results/m2-sensitivity-summary.csv` (extended-bins and male-only sensitivity runs).

In [5]:
ranked_df = read_results_csv("m2-aging-signal-ranked.csv")
elig_df = read_results_csv("m2-unit-eligibility.csv")
trio_df = read_results_csv("m2-calibration-trio-summary.csv")
sens_df = read_results_csv("m2-sensitivity-summary.csv")

n_evaluated = int(len(elig_df))
n_powered = int(elig_df["eligible"].astype(str).str.lower().eq("true").sum())
assert len(ranked_df) == n_powered == 17, (len(ranked_df), n_powered)
assert n_evaluated == 163, n_evaluated

ranking = {
    "ranked": clean_records(ranked_df),
    "n_units_evaluated": n_evaluated,
    "n_units_powered": n_powered,
    "eligibility_rule": ">=3 mice and >=50 cells per group (3m vs 18m); from results/m2-unit-eligibility.csv",
    "calibration_trio": clean_records(trio_df),
    "sensitivity": clean_records(sens_df),
}
write_json_asset(
    "ranking", ranking,
    "M2 mouse aging-signal ranking: 17 powered tissue x cell-type units (of 163 evaluated) ranked by pseudobulk DE gene count with bootstrap CIs, plus the calibration trio and sensitivity-run summaries.",
    ["m2-aging-signal-ranked.csv", "m2-unit-eligibility.csv", "m2-calibration-trio-summary.csv", "m2-sensitivity-summary.csv"],
)

  wrote app/data/ranking.json (5.3 KB) <- ['m2-aging-signal-ranked.csv', 'm2-unit-eligibility.csv', 'm2-calibration-trio-summary.csv', 'm2-sensitivity-summary.csv']


## 5. Gene-level program browser asset (M5)

Sources: `results/m5-gene-level-programs.csv` (312 program-gene × unit rows with log2FC, FDR, ribo/mito flag), `results/m5-mouse-program-catalog.csv` (per-unit program catalog incl. `frac_ribo_mito`).

In [6]:
gene_prog_df = read_results_csv("m5-gene-level-programs.csv")
catalog_df = read_results_csv("m5-mouse-program-catalog.csv")
assert len(gene_prog_df) == 312, len(gene_prog_df)

gene_programs = {
    "unit_gene_rows": clean_records(gene_prog_df),
    "catalog": clean_records(catalog_df),
}
write_json_asset(
    "gene_programs", gene_programs,
    "M5 gene-level program data: every program gene x unit row (log2FC, FDR, is_ribo_mito) and the per-unit program catalog with ribo/mito fractions.",
    ["m5-gene-level-programs.csv", "m5-mouse-program-catalog.csv"],
)

  wrote app/data/gene_programs.json (49.3 KB) <- ['m5-gene-level-programs.csv', 'm5-mouse-program-catalog.csv']


## 6. Conservation / prior-work agreement asset (M3 + M5 + M4)

Sources: `results/m3-agreement-table.csv` (our DE vs the TMS paper's published aging coefficients, per unit: Spearman ρ, sign concordance, verdict), `results/m5-divergence-table.csv` (conservation status per unit — the human arm is untestable with obtainable public data), `results/m4-human-power.csv` (human power grid; every scenario is Low-Powered at n=4 donors / SD≤16y).

In [7]:
agree_df = read_results_csv("m3-agreement-table.csv")
diverge_df = read_results_csv("m5-divergence-table.csv")
hpower_df = read_results_csv("m4-human-power.csv")

verdict_counts = agree_df["verdict"].value_counts().to_dict()
print("verdict counts:", verdict_counts)
assert verdict_counts.get("agree") == 13
assert verdict_counts.get("partial") == 4
assert verdict_counts.get("disagree", 0) == 0
assert diverge_df["status"].str.contains("untestable").all()

agreement = {
    "agreement": clean_records(agree_df),
    "verdict_counts": {k: int(v) for k, v in verdict_counts.items()},
    "divergence": clean_records(diverge_df),
    "human_power_grid": clean_records(hpower_df),
}
write_json_asset(
    "agreement", agreement,
    "M3 agreement with the TMS paper's published aging coefficients (13 agree / 4 partial / 0 disagree), M5 divergence table (human arm untestable with obtainable public data), and the M4 human power grid.",
    ["m3-agreement-table.csv", "m5-divergence-table.csv", "m4-human-power.csv"],
)

verdict counts: {'agree': 13, 'partial': 4}
  wrote app/data/agreement.json (36.3 KB) <- ['m3-agreement-table.csv', 'm5-divergence-table.csv', 'm4-human-power.csv']


## 7. Confounding + composition asset (M1)

Sources: `results/m1-sex-age-table.csv` (sex × age-group cell/mouse counts — the droplet arm is sex-confounded outside 3m vs 18m), `results/m1-composition-table.csv` (observed vs paper-reported cell-type proportions).

In [8]:
sex_age_df = read_results_csv("m1-sex-age-table.csv")
comp_df = read_results_csv("m1-composition-table.csv")

confound = {
    "sex_age": clean_records(sex_age_df),
    "composition": clean_records(comp_df),
}
write_json_asset(
    "confound", confound,
    "M1 sex-by-age confounding table (cells and mice per age group x sex) and the cell-type composition table vs the paper.",
    ["m1-sex-age-table.csv", "m1-composition-table.csv"],
)

  wrote app/data/confound.json (22.1 KB) <- ['m1-sex-age-table.csv', 'm1-composition-table.csv']


## 8. UMAP subsample asset (M1)

Source: `data/interim/m1-processed.h5ad` — `obs` (age, tissue, sex, cell_ontology_class) and `obsm['X_umap']`, read in backed mode (no full matrix load). Sampling: **seed 42**, per-tissue proportional allocation, **capped at 5,000 cells total**. Coordinates are rounded to 2 decimals (display decimation); the seed, the cap, and the exact per-tissue counts are recorded in the asset. Categoricals are stored as integer codes + category lists to keep the payload small.

In [9]:
import scanpy as sc

assert H5AD_PATH.exists(), f"processed object missing: {H5AD_PATH}"
adata = sc.read_h5ad(H5AD_PATH, backed="r")

obs = adata.obs[["age", "tissue", "sex", "cell_ontology_class"]].copy()
age_cats = list(obs["age"].cat.categories)
tissue_cats = list(obs["tissue"].cat.categories)
sex_cats = list(obs["sex"].cat.categories)
ct_cats = list(obs["cell_ontology_class"].cat.categories)

n_total = len(obs)
tissue_counts = obs["tissue"].value_counts()
# Per-tissue proportional allocation of the 5000-cell cap, largest-remainder to hit the cap exactly.
raw_quota = UMAP_CAP * tissue_counts / n_total
quota = np.floor(raw_quota).astype(int)
remainder = UMAP_CAP - int(quota.sum())
frac_order = (raw_quota - quota).sort_values(ascending=False).index
for t in frac_order[:remainder]:
    quota[t] += 1
assert int(quota.sum()) == UMAP_CAP

rng = np.random.default_rng(UMAP_SEED)
obs["_tissue_code"] = obs["tissue"].cat.codes.to_numpy()
tissue_code_to_name = dict(enumerate(tissue_cats))
pick_mask = np.zeros(n_total, dtype=bool)
for t in tissue_cats:
    idx = np.flatnonzero(obs["_tissue_code"].to_numpy() == tissue_cats.index(t))
    chosen = rng.choice(idx, size=int(quota[t]), replace=False)
    pick_mask[chosen] = True

X_umap = np.asarray(adata.obsm["X_umap"])[pick_mask]
sub = obs[pick_mask]
assert len(sub) == UMAP_CAP
adata.file.close()

umap_asset = {
    "meta": {
        "seed": UMAP_SEED,
        "cap": UMAP_CAP,
        "n_cells_total_in_object": int(n_total),
        "n_sampled": int(len(sub)),
        "sampling": "per-tissue proportional, seeded numpy.random.default_rng(42), without replacement",
        "coordinate_decimation": "x/y rounded to 2 decimals for display",
        "per_tissue_counts": {t: int(quota[t]) for t in tissue_cats},
    },
    "categories": {"age": age_cats, "tissue": tissue_cats, "sex": sex_cats, "cell_type": ct_cats},
    "x": [round(float(v), 2) for v in X_umap[:, 0]],
    "y": [round(float(v), 2) for v in X_umap[:, 1]],
    "age": [int(c) for c in sub["age"].cat.codes],
    "tissue": [int(c) for c in sub["tissue"].cat.codes],
    "sex": [int(c) for c in sub["sex"].cat.codes],
    "cell_type": [int(c) for c in sub["cell_ontology_class"].cat.codes],
}
write_json_asset(
    "umap_subsample", umap_asset,
    "M1 UMAP subsample: 5,000-cell seeded per-tissue proportional subsample of obsm['X_umap'] with age, tissue, sex, and cell-type codes.",
    [],  # non-CSV source; recorded manually below
)
register_asset(
    "umap_subsample", "data/umap_subsample.json",
    "M1 UMAP subsample: 5,000-cell seeded per-tissue proportional subsample of obsm['X_umap'] with age, tissue, sex, and cell-type codes.",
    [H5AD_PATH],
)

  wrote app/data/umap_subsample.json (132.1 KB) <- []


## 9. Figure copies for the app

The signature view renders the M6 figures directly; copies land in `app/figures/` so the app is a self-contained static tree. `figures/m5-housekeeping-guard.png` does not exist in this repo, so it is not copied (the recurrence figure is used in the conservation view instead).

In [10]:
FIGURE_COPIES = {
    "m6-signature-figure.png": "M6 signature figure: conserved program score vs life-stage-normalized age (mouse, human, NMR).",
    "m6-savina-reproduction.png": "M6 Savina 2022 reproduction: NMR DE-count vs |log2FC| cutoff curve.",
    "m6-positive-control-skin.png": "M6 positive control: whole-body mouse program score vs age.",
    "m5-program-recurrence.png": "M5 program recurrence across units.",
}
for fname, desc in FIGURE_COPIES.items():
    src = FIGURES_DIR / fname
    assert src.exists(), f"figure missing: {src}"
    dst = APP_FIGURES_DIR / fname
    shutil.copy2(src, dst)
    size = dst.stat().st_size
    assert size <= MAX_ASSET_BYTES, f"figure '{fname}' is {size} bytes > {MAX_ASSET_BYTES}"
    register_asset(f"figures/{fname}", f"figures/{fname}", desc, [src])
    print(f"  copied figures/{fname} -> app/figures/{fname} ({size/1024:.1f} KB)")

  copied figures/m6-signature-figure.png -> app/figures/m6-signature-figure.png (75.3 KB)
  copied figures/m6-savina-reproduction.png -> app/figures/m6-savina-reproduction.png (50.0 KB)
  copied figures/m6-positive-control-skin.png -> app/figures/m6-positive-control-skin.png (39.5 KB)
  copied figures/m5-program-recurrence.png -> app/figures/m5-program-recurrence.png (18.6 KB)


## 10. Manifest + budget audit

`app/data/manifest.json` maps every asset → source path(s) + SHA-256 of each source file. `scripts/check_app_assets.py` re-verifies this manifest on demand.

In [11]:
for key, meta in ASSETS.items():
    f = APP_DIR / meta["path"]
    assert f.exists(), f"asset file missing: {f}"
    meta["bytes"] = f.stat().st_size

manifest = {
    "generated_by": "notebooks/07-app-assets.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "project": "Conserved Aging Explorer",
    "budgets": {
        "max_asset_bytes": MAX_ASSET_BYTES,
        "max_app_data_total_bytes": MAX_DATA_TOTAL_BYTES,
        "pre_registered_outer_budgets": "<=25 MB per view, <=300 MB total (PLAN.md); far from binding at this data scale",
    },
    "umap_seed": UMAP_SEED,
    "assets": ASSETS,
}
with open(APP_DATA_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

# Budget audit
data_files = sorted(APP_DATA_DIR.glob("*.json"))
total = 0
for p in data_files:
    size = p.stat().st_size
    total += size
    assert size <= MAX_ASSET_BYTES, f"{p.name}: {size} bytes exceeds 1 MB cap"
    print(f"  {p.name:28s} {size/1024:8.1f} KB")
print(f"TOTAL app/data: {total/1024:.1f} KB (cap {MAX_DATA_TOTAL_BYTES/1024/1024:.0f} MB)")
assert total <= MAX_DATA_TOTAL_BYTES
fig_total = sum(p.stat().st_size for p in APP_FIGURES_DIR.glob("*.png"))
print(f"app/figures total: {fig_total/1024:.1f} KB (outside app/data budget, per-file cap still enforced above)")

  agreement.json                   36.3 KB
  confound.json                    22.1 KB
  gene_programs.json               49.3 KB
  manifest.json                     7.3 KB
  program_genes.json                2.6 KB
  ranking.json                      5.3 KB
  savina.json                       3.2 KB
  signature.json                    9.4 KB
  umap_subsample.json             132.1 KB
TOTAL app/data: 267.6 KB (cap 5 MB)
app/figures total: 183.3 KB (outside app/data budget, per-file cap still enforced above)


## 11. Golden-number assertions

Cross-check the exported assets against the headline numbers recorded in the milestone outputs, so a stale or corrupted CSV fails here rather than shipping to the app.

In [12]:
with open(APP_DATA_DIR / "signature.json") as f:
    sig = json.load(f)
stats_by_comp = {r["comparison"]: r for r in sig["statistics"]}

mw = stats_by_comp["Mouse (whole-body) aging slope"]
assert abs(mw["slope"] - 0.23495923946097935) < 1e-12 and mw["n"] == 23 and mw["p"] < 0.01
nmr = stats_by_comp["Naked mole-rat aging slope"]
assert abs(nmr["slope"] - 0.016674646055053994) < 1e-12 and nmr["n"] == 5 and nmr["p"] > 0.8
tost = stats_by_comp["NMR TOST equivalence (delta=0.059 = 25% of mouse slope)"]
assert tost["p"] > 0.3  # equivalence NOT established
diff = stats_by_comp["Mouse (whole-body) vs Naked mole-rat slope difference"]
assert 0.05 < diff["p"] < 0.08
perm = stats_by_comp["NMR aged-vs-young score difference (exact permutation, sample-level)"]
assert abs(perm["p"] - 0.8181818181818182) < 1e-12

gates_by_name = {g["gate"]: g for g in sig["gates"]}
assert abs(gates_by_name["coverage_nmr"]["value"] - 0.31413612565445026) < 1e-12
assert abs(gates_by_name["coverage_human"]["value"] - 0.7318007662835249) < 1e-12
assert abs(gates_by_name["tost_delta"]["value"] - 0.05873980986524484) < 1e-12

assert {r["common_name"]: r["max_lifespan_years"] for r in sig["lifespans"]} == {
    "House mouse": 4, "Naked mole-rat": 31, "Human": 122.5}

with open(APP_DATA_DIR / "program_genes.json") as f:
    pg = json.load(f)
assert pg["counts"] == {"n_program_up": 167, "n_program_down": 94, "n_conserved_up": 2, "n_conserved_down": 2}

with open(APP_DATA_DIR / "savina.json") as f:
    sv = json.load(f)
assert sv["curve"][0]["log2fc_cutoff"] == 0.1 and sv["curve"][-1]["log2fc_cutoff"] == 2.0
assert sv["positive_controls"][0]["gene"] == "Igfbp3"

with open(APP_DATA_DIR / "agreement.json") as f:
    ag = json.load(f)
assert ag["verdict_counts"] == {"agree": 13, "partial": 4}
assert all("untestable" in r["status"] for r in ag["divergence"])

with open(APP_DATA_DIR / "umap_subsample.json") as f:
    um = json.load(f)
assert um["meta"]["seed"] == 42 and um["meta"]["n_sampled"] == 5000
assert len(um["x"]) == len(um["y"]) == len(um["age"]) == 5000

print("ALL GOLDEN-NUMBER ASSERTIONS PASSED")

ALL GOLDEN-NUMBER ASSERTIONS PASSED


## 12. Outputs written

File path + SHA-256 + size for every output of this notebook (repo convention).

In [13]:
try:
    git_commit = subprocess.run(
        ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, capture_output=True, text=True
    ).stdout.strip()
except Exception:
    git_commit = "<unavailable>"

outputs = sorted(list(APP_DATA_DIR.glob("*.json")) + list(APP_FIGURES_DIR.glob("*.png")))
print(f"git commit at execution: {git_commit}")
for p in outputs:
    print(f"{p.relative_to(REPO_ROOT)}  {p.stat().st_size} bytes  sha256={sha256_of(p)}")
print(f"\nmanifest digest: sha256={sha256_of(APP_DATA_DIR / 'manifest.json')}")

git commit at execution: 441b803109e954e79d848b9f7e2629edd5b5b7c6
app/data/agreement.json  37135 bytes  sha256=c184c86024391581c5d0847b0fb91271d4a2678ccd58d942d603a886b55a2413
app/data/confound.json  22620 bytes  sha256=5a65af29147114753c71ca31ad3a32a848aef26135eac7a5c98070ce9e71a837
app/data/gene_programs.json  50489 bytes  sha256=4b89a2296021bb8d0f4feb77e00c1daac5aae36beda4fe0b68c51d4270bf85d7
app/data/manifest.json  7483 bytes  sha256=e9a52d812bc6cdce66ce961671a5ef7676301cb3320b8998655b5bb638168140
app/data/program_genes.json  2633 bytes  sha256=7abc48cd12a3ca5cd3c9f76003337dd4410e944153f51b2ea05ce662c93dc01c
app/data/ranking.json  5392 bytes  sha256=c78acb69cce423d8825618d299208023e218095092c4d4375162414e315af313
app/data/savina.json  3309 bytes  sha256=02578c32ea6006f58a366e59f1e7c8b6a4a71ab2d3f630077eec8bce6748bc5a
app/data/signature.json  9641 bytes  sha256=61ca7e3e983b84a3e1c91afa6f090d11b3729ee30dc3df11d8d0598695e54aa9
app/data/umap_subsample.json  135281 bytes  sha256=e88cb42